   When you invoke `strtol`, the engine executes FIVE DISTINCT PHASES in a 
   single sequential pass across your string:

   1. WHITESPACE SKIP: It discards any leading whitespace characters (spaces,
      tabs, newlines).
   2. SIGN ACKNOWLEDGMENT: It notes an optional positive (`+`) or negative (`-`)
      sign.
   3. BASE ASSESSMENT: If base is specified (e.g., `10`or `16`), it enforces
      that mathematical universe. If `base` is 16, it automatcailly steps past
      a leading `0x` or `0X` if present.
      - If `base` is 0, it switches on AUTO-DETECT MODE: a leading `0x` sets 
        the base to 16, a leading `0` sets it to 8 (octal), amd anything else 
        defaults to 10 (decimal).
   4. DIGIT ACCUMULATION: It consumes valid characters matching the base... 
      stops at first non-digit... base 16, treats `a` to `f` as valid numeric digits...
   5. THE `endptr` MUTATION (The Core Magic): You pass `strtol` the address of
      a local pointer (`&end`). The engine modifies your pointer to point to
      the exact memory address where it hit the first invalid character...

DECODING THE THREE `endptr` STATES
   ... accuratly deduce the health of your string:
   - PERFECT SUCCESS (`*end == '\0'`): The engine scanned the entire string and
     cleanly stopped at the invisible null terminator. The input was a pure, 
     clean number.
   - PARTIAL GARBAGE (`*end != '\0' && end != str`): The engine manages to parse 
     a number at the beginning, but hit illegal characters midway ... 
   - ABSOLUTE FAILURE (`end == str`): The engine couldn't find a single valid 
     digit to convert (e.g., `"abc"`). The address in `end` hasn't advanced
     at all from the starting address.


---
Q1:
   

---

THE `strncmp` MECHANICS
   `strncmp` (String Number Compare) compares two strings, but halts after a
   specified number of characters (n). This makes it the perfect tool for prefix
   checking, becuase it allows you to inspect the beginning of a string while
   completely ignoring whatever text or numbers follow it.


THE BLUEPRINT
```c
#include <string.h>

int strncmp(const char *s1, const char *s2, size_t n);
```
   - `s1`: The string you are inspecting (e.g., user input like `"offset:0x1F"`)
   - `s2`: The exact prefix target you are looking for (e.g., `"offset:"`)
   - `n` : The exact nuber of characters to compare


THE RETURN VALU TRAP
   The most common mistake with `strncmp` is treating it like a boolean 
   true/false.
   - `strncmp` returns `0` if the strings are a perfect match up to $n$
     characters (meaning there are ZERO DIFFERENCES).
   - It returns a non-zero value if they do not match.


HOW TO USE FOR YOUR PREFIX CHECK
   Since `"offset:"` contains exactly 7 characters...

```c
#include <stdio.h>
#include <string.h>

void check_prefix(const char *s) {
    // Compare only the first 7 characters of 's' against "offset:"
    if (strncmp(s, "offset:", 7) == 0) {
        printf("Match found! The string starts with 'offset:'\n");
    } else {
        printf("Invalid syntax\n");
    }
}
```


---

```c
#include <string.h> 
#include <stdio.h>

int parse_offset(const char *s, long *out_val) {
    const int len = strlen("offset:");
    if (strncmp(s, "offset:", len) != 0) {
        printf("Invalid syntax\n");
        return -1;
    }
    char *end;
    long val = strtol(s + len, &end, 0);
    if (*end != '\0' || end == (s + len)) {
        printf("Invalid syntax\n");
        return -1;
    }
    *out_val = val;
    return 0;
}
```


---

```c
#include <errno.h>
#include <stdio.h>
#include <strlib.h>

int safe_parse_long(const char *s, long *out_val) {
    char *endptr;
    errno = 0;
    long val = strtol(s, &endptr, 10);
    if (endptr == s) {
        fprintf(stderr, "ERROR: String is empty or has no digits.\n");
        return -1;
    } else if (*endptr != '\0') {
        fprintf(stderr, "ERROR: Trailing garbage detected in string.\n");
        return -2;
    } else if (errno == ERANGE) {
        fprintf(stderr, "ERROR: Number under/over-flowed long limits.\n");
        return -3;
    } else {
        printf("SUCCESS: String perfectly clean and fits inside a long.\n");
        *out_val = val;
        return 0;
    }
}

```

---

THE CONCEPT BREAKDOWN
1. Formatted Text Scanning (`sscanf` and `fscanf`)
   Think of these as the reverse of `printf`. Instead of layout out data into
   text, they dissect text back into raw primitives using matching patterns.
   - `sscanf(buffer, format, ...)` reads from an existing string array in memory.
   - `fscanf(fp, format, ...)` reads directly from an open file pointer stream.
   - THE RETURN VALUE LAW: Both functions return the NUMBER OF SUCCESSFULLY
     matched and assigned items. Alwas check this value! If you are scanning for
     two integers (`"%d %d"`), make sure the function returns exactly `2`. If
     it returns `0` or `1` the input data is malformed.
   - DOUBLE FLOATING-POINT TRAP: While `printf` uses `%f` for both float and
     double, `sscanf`/`fscanf` require `%lf` (long float) to correctly parse 
     data into a 64-bit `double`. Using `%f` for a double will silently corrupt
     your memory.


- "r"/"rb"
- "w"/"wb"
- "a"/"ab"
    - Suffix `b` - Designates BINARY MODE, which prevents the OS from translating
      line endings (`\r\n` vs `\n`).


---
   - SAFETY CHECKING: `fopen` will return `NULL` if a file is missing, permissions
     are denied, or the disk is locked.  ... Never read or write to a `NULL`
     file pointer... will trigger immediate segmentation fault...
   - RESOURCE ALLOCATION: ... Every `fopen` must have a corresponding `fclose`.
     Leaving files open leaks file descriptors back to the OS, which can eventually 
     ... 


3. BLOCK BINARY I/O (`fread` and `fwrite`)
   When building emulators or assemblers, you don't save raw text. You save and
   read raw, binary data blocks (like whole structs or RAM buffers) directly to
   disk.
```c
size_t fread(void *ptr, size_t size, size_t n, FILE *fp);
```
   - `ptr`: The memory address where the data should be written to.
   - `size`: The size of a single element `sizeof(uint32_t)` or `sizeof(struct Instruction)`
   - `n`: The maximum number of elements you want to read/write.
   - RETURN VALUE: Returns the number of ELEMENTS successfully handled, not the
     number of bytes. 


---
4. CODE ASSERTIONS  (`#include <assert.h>`)
   The `assert(condition)` macro is a developer-only panic switch. If the 
    expression evalates to false, your program halts, instantly printing the 
    exact line number and filename to terminal.
    - PROPRER USE: Use assertions only for DEVELOPER ASSUMPTIONS and CODE 
      INVARIANTS... (e.g., `assert(ram_ptr != NULL);`... inside an instruction
      executor loop...)
    - IMPROPER USE: Never use assertions for RUNTIME ENVIRONMENTS or USER
      INPUT ERRORS (e.g., checking if `fopen` succeeded, or checking if 
      `argc == 2`). If a file is missing... should print elegant error message
      ... `fprintf(stderr, " ... \n");`, not crash hard in front of end user...

---

```c
#include <stdio.h>

int clone_text_file(const char *src_path, const char *dest_path) {
    FILE *fp_src = fopen(src_path, "r");
    FILE *fp_dest = fopen(dest_path, "w");

    if (fp_src == NULL || fp_dest == NULL) {
        fclose(fp_src);
        fclose(fp_dest);    
        fprintf(stderr, "ERROR: Invalid file path for src/dest file.\n");
        return -1;
    }

    int c_src;
    int c_dest = '1';
    while ((c = getc(fp_src)) != EOF && (c_dest != EOF)) {
        c_dest = fputc(c, fp_dest);
    }

    fclose(fp_src);
    fclose(fp_dest);
    return 0;
}
```

int clone_text_file(const char *src_path, const char *dest_path) {
    FILE *fp_src = fopen(src_path, "r");
    if (fp_src == NULL) {
        fprintf(stderr, "ERROR: Unable to open source file.\n");
        return -1;
    }

    FILE *fp_dest = fopen(dest_path, "w");
    if (fp_dest == NULL) {
        fprintf(stderr, "ERROR: Unable to open destination file.\n");
        fclose(fp_src);
        return -1;
    }

    int c;
    while ((c = fgetc(fp_src)) != EOF) {
        if (fputc(c, fp_dest) == EOF) {
            fprintf(stderr, "ERROR: Disk write failure during cloning.\n");
            fclose(fp_src);
            fclose(fp_dest);
            return -1;
        }
    }

    fclose(fp_src);
    fclose(fp_dest);
    return 0;
}

---

getchar/getc/fgetc/(fputc)  -> read/write raw characters
fgets                       -> read one whole line into a string buffer
sscanf                      -> parse structured values from an existing string
strtol                      -> parse one integer very carefully
fread                       -> read raw binary bytes

---

   `sscanf` means STRING SCANF. It does not read from the keyboard/file. It 
   reads from the keyboard/file. It reads from a `char *` string you already 
   have.

---
```c
sscanf(input_string, "format", addresses_to_store_result);
```

EXAMPL 1: parse one int

```
char line[] = "42";
int x;

if (sscanf(line, "%d", &x) == 1) {
    printf("x = %d\n", x);
}
```


```c
char *line = "42";
int x;

if (sscanf(line, "%d", &x) == 1) {
    printf("x = %d", x);
}
```


E2: Parse two ints
```c
char *line = "12 34"
int a, b;

if (sscanf(line, "%d %d", a, b) == 2) {
    printf("a = %d, b = %d\n", a, b);
}
```


E3: parse a line from a file
```c
#include <stdio.h>
#include <stdlib.h>

#define MAX_SIZE 1024

int main() {
    char *buf = malloc(sizeof(char) * MAX_SIZE);
    while (fgets(buf, MAX_SIZE, fp) != NULL) {
        double x;

        if (sscanf(buf, "%lf", &x) == 1) {
            sum += x;
        } else {
            fprintf(stderr, "ERROR: bad line '%s'\n", buf);
        }
    }
    free(buf);
}
```







```c
char *line = "x=10 y=20";
int x, y;

if (sscanf(line, "x=%d y=%d",&x, &y) == 2) {
    printf("Coordinates = (%d, %d)\n", x, y);
}
```

---

```c
char *line = "Alice 19";
char *name = malloc(sizeof(char) * 20);
int  age;

if (sscanf(line, "%19s %d", &name, &age) == 2) {
    printf("%s is of %d years old this year.\n", name, age);
}
```


sscanf is useful after fgets
getchar is for raw character streams
strtol is for careful single-number parsing
fread is for binary bytes


---

```c
#include <stdlib.h>

int parse_gps_coordinates(const char *buf, double *lat, double *lon) {
    if (sscanf(buf, "GPS: [%lf, %lf]", lat, lon) != 2) {
        return -1;
    }
    return 0;       // Perfect Match
}

```

```c
#include <stdio.h>

int sum_score_file(const char* filename) {
    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        return -1;
    }

    char buf[256];
    char temp_name[64];
    int score;
    int total = 0;

    while (fgets(buf, sizeof(buf), fp) != NULL) {
        if ((sscanf(buf, "%63s %d", temp_name, &score)) == 2) {
            total += score;
        }
    }

    fclose(fp);
    return total;
}
```

---


```c
#include <stdio.h>
#include <assert.h>

typedef struct {
    uint32_t opcode,
    uint8_t target_register
} Instruction;

int archive_instructions(const char *path, const Instruction *arr, 
                         size_t count) {
    assert(arr != NULL);
    assert(count > 0);

    FILE *fp = fopen(path, "w");
    if (fp == NULL) {
        return -1;
    }
    
    size_t items_written = fwrite(arr, sizeof(Instruction), count, fp);
    
    fclose(fp);

    if (items_written != count) {
        return -1;      // Disk full or I/O error
    }
    return 0;
}

```



```c
size_t elements_read = fead(buffer, sizeof(buffer[0]), count, file_pointer);
size_t written = fwrite(buffer, sizeof(buffer[0]), count, file_pointer);
```

---

```c
#include <stdio.h>

int load_binary_to_ram(const char *filename, uint8_t *ram, 
                       size_t max_ram_size) {
    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        return -1;
    }

    size_t bytes_read = fread(ram, sizeof(ram[0]), max_ram_size, fp);
    if (fgetc(fp) != EOF) {
        fprintf(stderr, "ERROR: Binary file exceeds maximum RAM allocation,\n");
        fclose(fp);
        return -1;
    }

    fclose(fp);
    return (int)bytes_read;
}


```

---

```c
typedef struct person person;

typedef struct {
    int age;
    person *next;       // PERFECTLY LEGAL: 'person' was declared above!
};
```


```c
typedef struct {
    char name[80];
    int age;
} person;

void birthday(struct person *p) {
    assert(p != NULL);
    p->age++;
}
```

```c
#include <stdio.h>

int count_chars(const char *filename) {
    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        return -1;
    }

    int total = 0;
    int c;
    while ((c = fgetc(fp)) != EOF) {
        total++;
    }

    fclose(fp);
    return total;
}

```

```

```